In [0]:
source_path = "/Volumes/fraudradar/source/fraud_watchlist/source_data/"

In [0]:
dbutils.fs.ls(source_path)

In [0]:
input_data = (spark.readStream.format("cloudFiles") 
             .option("cloudFiles.format", "json") 
             .option("cloudFiles.schemaLocation", "/Volumes/fraudradar/source/fraud_watchlist/schema/") 
             .option("cloudFiles.inferColumnTypes", "true")
             .load(source_path))

In [0]:
from pyspark.sql import functions as F

parsed_df = input_data.select(
    "*",
    F.col("_metadata.file_path").alias("file_path"),
    F.current_timestamp().alias("ingest_ts")
)

In [0]:
streaming_query = (parsed_df.writeStream.format("delta")
.outputMode("append")
.option("checkpointLocation", "/Volumes/fraudradar/source/fraud_watchlist/checkpoint/")
.trigger(availableNow=True)
.toTable("fraudradar.bronze.fraud_watchlist_batch_test"))

In [0]:
%sql
SELECT * FROM fraudradar.bronze.fraud_watchlist_batch_test 